MRP

In [8]:
import numpy as np

np.random.seed(0)
p = [
    [0.9,0.1,0.0,0.0,0.0,0.0],
    [0.5,0.0,0.5,0.0,0.0,0.0],
    [0.0,0.0,0.0,0.6,0.0,0.4],
    [0.0,0.0,0.0,0.0,0.3,0.7],
    [0.0,0.2,0.3,0.5,0.0,0.0],
    [0.0,0.0,0.0,0.0,0.0,1.0],
]
p = np.array(p)
rewards = [-1,-2,-2,10,1,0]
gamma = 0.5
def compute_return(start_index,chain,gamma):
    # start_index 标识从chain中第start_index作为起点
    G = 0
    for i in reversed(range(start_index,len(chain))):
    # 反转序列
        G = gamma*G  + rewards[chain[i] - 1]
    return G
chain = [1,2,3,6]
start_index = 0
G = compute_return(start_index,chain,gamma)
G

-2.5

In [9]:
def compute(P,rewards,gamma,states_nums):
    rewards = np.array(rewards).reshape((-1,1))
    value = np.dot(np.linalg.inv(np.eye(states_nums,states_nums)-gamma*P),rewards)
    # np.linalg.inv 求逆
    return value
V = compute(p,rewards,gamma,6)
V

array([[-2.01950168],
       [-2.21451846],
       [ 1.16142785],
       [10.53809283],
       [ 3.58728554],
       [ 0.        ]])

MDP

In [10]:
S = ["s1","s2","s3","s4","s5"] # 状态集合
A = [
    "保持s1",
    "前往s1",
    "前往s2",
    "前往s3",
    "前往s4",
    "前往s5",
    "概率前往"
] 
P = {
    "s1-保持s1-s1":1.0,"s1-前往s2-s2":1.0,
    "s2-前往s1-s1":1.0,"s2-前往s3-s3":1.0,
    "s3-前往s4-s4":1.0,"s3-前往s5-s5":1.0,
    "s4-前往s5-s5":1.0,"s4-概率前往-s2":0.2,
    "s4-概率前往-s3":0.4,"s4-概率前往-s2":0.4
}
R = {
    "s1-保持s1":-1,"s1-前往s2":0,
    "s2-前往s1":-1,"s2-前往s3":-2,
    "s3-前往s4":-2,"s3-前往s5":0,
    "s4-前往s5":10,"s4-概率前往":1
}
gamma = 0.5
MDP = (S,A,P,R,gamma)
# 策略1
pi_1 = {
    "s1-保持s1":0.5,"s1-前往s2":0.5,
    "s2-前往s1":0.5,"s2-前往s3":0.5,
    "s3-前往s4":0.5,"s3-前往s5":0.5,
    "s4-前往s5":0.5,"s4-概率前往":0.5
}
# 策略2
pi_2 = {
    "s1-保持s1":0.6,"s1-前往s2":0.4,
    "s2-前往s1":0.3,"s2-前往s3":0.7,
    "s3-前往s4":0.5,"s3-前往s5":0.5,
    "s4-前往s5":0.1,"s4-概率前往":0.9
}

def join(str1,str2):
    return str1 + "-" + str2

In [11]:
gamma = 0.5
p_from_mdp_to_mrp = [
    [0.5,0.5,0.0,0.0,0.0],
    [0.5,0.0,0.5,0.0,0.0],
    [0.0,0.0,0.0,0.5,0.5],
    [0.0,0.1,0.2,0.2,0.5],
    [0.0,0.0,0.0,0.0,1.0]
]
p_from_mdp_to_mrp = np.array(p_from_mdp_to_mrp)
R_from_mdp_to_mrp = [-0.5,-1.5,-1.0,5.5,0]
v = compute(p_from_mdp_to_mrp,R_from_mdp_to_mrp,gamma,5)
v

array([[-1.22555411],
       [-1.67666232],
       [ 0.51890482],
       [ 6.0756193 ],
       [ 0.        ]])

In [12]:
def sample(MDP,pi,timemax,number):
    """ 采样 """
    S,A,P,R,gamma = MDP
    episodes = []
    for _ in range(number):
        episode = []
        timestep = 0
        s =  S[np.random.randint(4)] # 选取一个初始状态
        while s !="s5" and timestep<=timemax:
            timestep+=1
            rand,temp = np.random.rand(),0
            for a_opt in A:
                temp += pi.get(join(s,a_opt),0)
                if temp > rand:
                    a = a_opt
                    r = R.get(join(s,a_opt),0)
                    break
            rand,temp = np.random.rand(),0
            for s_opt in S:
                temp +=P.get(join(join(s,a),s_opt),0)
                if temp >rand:
                    s_next = s_opt
                    break
            episode.append((s,a,r,s_next))
            s = s_next
        episodes.append(episode)
    return episodes

episodes = sample(MDP=MDP,pi=pi_1,timemax=20,number=5)
print("第一条序列\n",episodes[0])
print("第二条序列\n",episodes[1])
print("第五条序列\n",episodes[4])


第一条序列
 [('s1', '前往s2', 0, 's2'), ('s2', '前往s3', -2, 's3'), ('s3', '前往s5', 0, 's5')]
第二条序列
 [('s4', '概率前往', 1, 's5')]
第五条序列
 [('s2', '前往s3', -2, 's3'), ('s3', '前往s4', -2, 's4'), ('s4', '前往s5', 10, 's5')]


In [13]:
def MC(episodes,V,N,gamma):
    for episode in episodes:
        G = 0
        for i in range(len(episode)-1,-1,-1):
            (s,a,r,s_next) = episode[i]
            G = r +gamma*G
            N[s] = N[s]+1
            V[s] = V[s] + (G - V[s])/N[s]

episodes = sample(MDP=MDP,pi=pi_1,timemax=20,number=1000)
gamma = 0.5
V = {"s1":0,"s2":0,"s3":0,"s4":0,"s5":0} 
N = {"s1":0,"s2":0,"s3":0,"s4":0,"s5":0}
MC(episodes=episodes,V=V,N=N,gamma=gamma)
print("使用蒙特卡洛方法计算 MDP 的状态为\n",V)

使用蒙特卡洛方法计算 MDP 的状态为
 {'s1': -1.2316472894818067, 's2': -1.7062291203123139, 's3': 0.41620357293751353, 's4': 5.4522445946931795, 's5': 0}


In [ ]:
def occupancy(episodes,s,a,timestep_max,gamma):
    """计算占用度量"""
    rho = 0
    total_tomes = np.zeros(timestep_max)
    occur_times = np.zeros(timestep_max)

    for episode in episodes:
        for i in range(len(episode)):
            (s_opt,a_opt,r,s_next) = episode[i]
            total_tomes[i]+=1
            if s_opt==s and a_opt == a:
                occur_times[i]+=1
    for i in reversed(range(timestep_max)):
        if total_tomes[i]:
            rho += gamma**i*(occur_times[i]/total_tomes[i])
    
    return (1-gamma)*rho 

gamma = 0.5
episodes_1 = sample(MDP=MDP,pi=pi_1,timemax=1000,number=1000)
episodes_2 = sample(MDP=MDP,pi=pi_2,timemax=1000,number=1000)

rho_1 = occupancy(episodes=episodes_1,s="s4",a="概率前往",timestep_max=1000,gamma=gamma)
rho_2 = occupancy(episodes=episodes_2,s="s4",a="概率前往",timestep_max=1000,gamma=gamma)

print(rho_1,rho_2)

0.09316986097482605 0.1945921379349892
